Grupo H - Dataset Películas

Analizamos el Dataset objeto de nuestro estudio, de forma de lograr predecir el éxito (rating) de una película a estrenar, de acuerdo a su género.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/raw/final_dataset.csv")
df

A contiuación eliminamos las columnas detalladas debajo dado que no aportaban al objeto de nuestro análisis. Las mismas refieren a textos que describen caracterìsticas especìficas de cada pelìculas, que no agregan valor a efectos de nuestra predicción.

In [ ]:
df = df.drop(columns=["awards_content"])

df = df.drop(columns=["description"])
df = df.drop(columns=["movie_link"])
df = df.drop(columns=["mpa"])
df = df.drop(columns=["opening_weekend_gross"])
df = df.drop(columns=["gross_worldwide"])
df = df.drop(columns=["gross_us_canada"])
df = df.drop(columns=["filming_locations"])
df

Revisamos que no tuviéramos id duplicados, de forma de asegurarnos que no estábamos trabajando con películas repetidas.

In [ ]:
df = df.drop_duplicates(subset="id")
df

En un principio consideramos que a partir de Budget podíamos construir una buena predicción acerca del éxito de una película; sin embargo no contamos con datos en el 76% de las filas, por lo que la desestimamos

In [ ]:
df["budget"]

In [ ]:
df=df.drop(columns=["budget"])
df

Utilizamos la columna de Género para poder avanzar en una relación que nos permita predecir que tan exitosa será la película a estrenar de acuerdo a su género

In [ ]:
df["genres"]

Para poder usar la columna Género como parte del modelo, utilizamos la primer categoría de género, para ello limpiamos la columna, eliminando los corchetes y las comas.

In [ ]:
import ast
def primer_genero(x):
    if pd.isna(x):
        return None

    if isinstance(x, list):
        return x[0] if len(x) > 0 else None

    if isinstance(x, str):
        try:
            lista = ast.literal_eval(x)
            if isinstance(lista, list):
                return lista[0] if len(lista) > 0 else None
        except (ValueError, SyntaxError):
            return x.split(",")[0].strip()

    return None

df["genres"] = df["genres"].apply(primer_genero)
df


Utilizamos la columna de Rating para poder avanzar en una relación que nos permita predecir que tan exitosa será la película a estrenar considerando su género.

In [ ]:
df["rating"]


En un principio consideramos que a partir de la columna "Votes" podíamos construir una buena predicción acerca del éxito de una película; sin embargo no contamos con datos en el 12% de las filas, por lo que la desestimamos

In [ ]:
df["votes"]

In [ ]:
df=df.drop(columns=["votes"])
df

En un principio consideramos que a partir de "Meta Score" podíamos construir una buena predicción acerca del éxito de una película; sin embargo no contamos con datos en el 75% de las filas, por lo que la desestimamos

In [ ]:
df["méta_score"]

In [ ]:
df=df.drop(columns=["méta_score"])
df

In [ ]:
rating_por_genero = df.groupby("genres")["rating"].mean().sort_values(ascending=False)

print(rating_por_genero)
df


Graficamos la relación rating / género
Dado que el eje y se volvió ilegible dada la cantidad de géneros, optamos por trabajar por los 20 géneros con mayor rating.

Tomamos los 20 géneros de mayor rating, y realizamos la gráfica rating vs género.

In [ ]:
rating_por_genero = (
    df.groupby("genres")["rating"]
      .mean()
      .nlargest(20)
      .sort_values(ascending=True)
)

# Crear gráfico
plt.figure(figsize=(10, 7))

barras = plt.barh(
    rating_por_genero.index,
    rating_por_genero.values,
    color="steelblue"
)

plt.xlabel("Rating promedio")
plt.ylabel("Género")
plt.title("Top 20 géneros con mayor rating promedio")

# Mostrar el valor en cada barra
for barra in barras:
    valor = barra.get_width()
    plt.text(
        valor + 0.02,
        barra.get_y() + barra.get_height() / 2,
        f"{valor:.2f}",
        va="center"
    )

plt.tight_layout()
plt.show()

Buscamos establecer una correlación entre Duración de la película y el rating.
Para ello buscamos trabajar sobre la columna Duration, y luego de varios pasos, la expresamos en minutos

In [ ]:
df['duration']
df


In [ ]:
horas = df["duration"].str.extract(r"(\d+)\s*h").fillna(0).astype(int)
df

In [ ]:
minutos = df["duration"].str.extract(r"(\d+)\s*m").fillna(0).astype(int)
df


In [ ]:
df["duration_minutes"] = (horas * 60) + minutos
df

Buscamos establecer correlación entre Duracion en minutos de las películas y el Rating,una vez obtenida la correlación, la mostramos mediante gráfica

In [ ]:
correlacion = df["duration_minutes"].corr(df["rating"])
print(f"Coeficiente de correlación de Pearson: {correlacion:.2f}")
df


In [ ]:
import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

# Calcular correlación de Pearson

correlacion = df["duration_minutes"].corr(df["rating"])

# Crear gráfico

plt.figure(figsize=(10, 6))

sns.regplot(

    data=df,

    x="duration_minutes",

    y="rating",

    scatter_kws={"alpha": 0.4},

    line_kws={"linewidth": 2}

)

# Mostrar correlación

plt.text(

    0.03,

    0.95,

    f"Correlación de Pearson: {correlacion:.2f}",

    transform=plt.gca().transAxes,

    fontsize=11,

    verticalalignment="top"

)

# Título y etiquetas

plt.title(

    "Relación entre la duración y el rating",

    fontsize=14

)

plt.xlabel(

    "Duración (minutos)",

    fontsize=11

)

plt.ylabel(

    "Rating",

    fontsize=11

)

# Tamaño de los valores de los ejes

plt.xticks(fontsize=10)

plt.yticks(fontsize=10)

# Rating de 0 a 10

plt.ylim(0, 10.5)

# Ajustar espacios

plt.tight_layout()

plt.show()

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

# Calcular correlación de Pearson

correlacion = df["duration_minutes"].corr(df["rating"])

# Crear gráfico

plt.figure(figsize=(10, 6))

sns.regplot(

    data=df,

    x="duration_minutes",

    y="rating",

    scatter_kws={"alpha": 0.4},

    line_kws={"linewidth": 2}

)

# Mostrar correlación

plt.text(

    0.03,

    0.95,

    f"Correlación de Pearson: {correlacion:.2f}",

    transform=plt.gca().transAxes,

    fontsize=11,

    verticalalignment="top"

)

# Título y etiquetas

plt.title(

    "Relación entre la duración y el rating",

    fontsize=14

)

plt.xlabel(

    "Duración (minutos)",

    fontsize=11

)

plt.ylabel(

    "Rating",

    fontsize=11

)

# Tamaño de los valores de los ejes

plt.xticks(fontsize=10)

plt.yticks(fontsize=10)

# Rating de 0 a 10

plt.ylim(0, 10.5)

# Ajustar espacios

plt.tight_layout()

plt.show()


# Interpretación del resultado
 Si el coeficiente correlacion > 0.4, existe una correlación positiva moderada /fuerte, por lo que las películas más largas tienden a tener mejor puntuación rating.
Sin embargo, el coeficiente de correlación de Pearson en nuestro análisis es 0.16, por lo que la correlación es muy débil o casi inexistente."
 Concluimos que la duración de la película no parece influir en su calificación.

In [ ]:
df.to_csv("../data/processed/final_dataset_limpio.csv", index=False)